## **The following visuals were created with the assistance of AI**

### To fully communicate the incredible increase in the Housing Price Index vs Median Income I feel that animations of previous visuals tell the story better than usign the functions in the sql_and_visuals notebook. These animations provide a clear picture of the increasing challenges facing prospective homeowners today

Import libraries

In [14]:
import pandas as pd 
import sqlite3
import plotly.express as px



Open SQLite 3 database hpi_v_medic.db and create cursor object to execute commands

In [15]:
conn = sqlite3.connect("hpi_v_medinc.db")
cursor = conn.cursor()

Function to run queries

In [16]:
def query(query: str):
    return pd.read_sql(query, conn)

Created dictionary of states and specific two letter abbreviations for the Plotly choropleth map that will be used below as full state names cannot be used. I will also utilize this dictionary in the scatterplot for better readability. 

In [17]:
state_map = {
    'Alabama': 'AL','Alaska': 'AK','Arizona': 'AZ','Arkansas': 'AR',
    'California': 'CA','Colorado': 'CO','Connecticut': 'CT','Delaware': 'DE',
    'Florida': 'FL','Georgia': 'GA','Hawaii': 'HI','Idaho': 'ID',
    'Illinois': 'IL','Indiana': 'IN','Iowa': 'IA','Kansas': 'KS',
    'Kentucky': 'KY','Louisiana': 'LA','Maine': 'ME','Maryland': 'MD',
    'Massachusetts': 'MA','Michigan': 'MI','Minnesota': 'MN','Mississippi': 'MS',
    'Missouri': 'MO','Montana': 'MT','Nebraska': 'NE','Nevada': 'NV',
    'New Hampshire': 'NH','New Jersey': 'NJ','New Mexico': 'NM','New York': 'NY',
    'North Carolina': 'NC','North Dakota': 'ND','Ohio': 'OH','Oklahoma': 'OK',
    'Oregon': 'OR','Pennsylvania': 'PA','Rhode Island': 'RI','South Carolina': 'SC',
    'South Dakota': 'SD','Tennessee': 'TN','Texas': 'TX','Utah': 'UT',
    'Vermont': 'VT','Virginia': 'VA','Washington': 'WA','West Virginia': 'WV',
    'Wisconsin': 'WI','Wyoming': 'WY'
}

Function which includes a query to compare HPI to Median Income and display the results in an animated scatterplot to clearly see which states are more affordable to live in and which are the most expensive. Called in dictionary from above to display two letter abbreviations for each state to improve readability of scatterplot. The animated scatterplot may be the best illustration of the significant jump in HPI. Press play on visual to see animation. Notice that when the animation reaches 2024 there aren't any states where Median Income increase is greater that HPI. This is the first year for the time frame of this project where that happened.

In [18]:
df_all = query("""
    SELECT
        h.place_name,
        h.index_nsa AS hpi,
        m.change AS income_index,
        h.yr AS year
    FROM hpi h
    JOIN medinc m
      ON h.place_name = m.place_name
     AND h.yr = m.yr
""")

df_all['place_id'] = df_all['place_name'].map(state_map)

fig = px.scatter(
    df_all,
    x="income_index",
    y="hpi",
    animation_frame="year",
    text="place_id",
    labels={
        "income_index": "Income Growth (%)",
        "hpi": "HPI Growth (%)",
        "year": "Year"
    },
    range_x=[0, 350],
    range_y=[0, 650]
)

fig.update_traces(
    textposition="top center",
    marker=dict(size=10, opacity=0.7, color="#4f4da7")
)

fig.add_shape(
    type="line",
    x0=0, y0=0,
    x1=350, y1=350,
    line=dict(
        color="#A5A74D",
        width=2,
        dash="dash"
    )
)


fig.update_layout(
    width=1100,
    height=700
)

fig.update_layout(
    title="HPI vs Income Over Time",
    showlegend=False
)

fig.show()


Function to call animated cholopleth map of US. Called in dictionary from above to create extra column in df 'place_id' using the two letter state abbreviations that corresdond to the full state name. When hovering on a state the state's full name will be displayed and the gap between HPI and median income for the year chosed. Dark blue represents the lowest gap while dark red represents the highest gap

In [24]:
def hpi_vs_income_choropleth_animated():

    df = query("""
        SELECT
            h.place_name,
            h.index_nsa AS hpi,
            m.change AS income_index,
            (h.index_nsa - m.change) AS gap,
            h.yr AS year
        FROM hpi h
        JOIN medinc m
            ON h.place_name = m.place_name
            AND h.yr = m.yr
    """)

    
    df['place_id'] = df['place_name'].map(state_map)

    
    fig = px.choropleth(
        df,
        locations='place_id',
        locationmode='USA-states',
        scope='usa',
        color='gap',
        animation_frame='year',   
        color_continuous_scale='Rainbow',
        range_color=[0, 400],
        hover_name='place_name',
        hover_data={
            'place_id': False,
            'gap': ':.2f'
        },
        title='Housing vs Income Gap by State Over Time'
    )

    fig.update_layout(
        height=500,
        width=900,
        title_x=0.5
    )

    return fig

Call function then press play on the visual

In [25]:
fig = hpi_vs_income_choropleth_animated()
fig.show()

Function which includes a query that returns animated grouped bar chart showing HPI percent increase and Median Income percent increase for the years 1991 to 2024

In [21]:
def hpi_vs_income_bar_animated():

    df = query("""
        SELECT
            h.place_name,
            h.index_nsa AS hpi,
            m.change AS income_index,
            h.yr AS year
        FROM hpi h
        JOIN medinc m
            ON h.place_name = m.place_name
            AND h.yr = m.yr
    """)

    
    df = df.sort_values(['year', 'hpi'], ascending=[True, False])

    
    df_long = df.melt(
        id_vars=['place_name', 'year'],
        value_vars=['hpi', 'income_index'],
        var_name='metric',
        value_name='value'
    )

    
    fig = px.bar(
        df_long,
        x='place_name',
        y='value',
        color='metric',
        barmode='group',
        animation_frame='year',   
        title='HPI vs Income by State Over Time',
        labels={
            'value': 'Percent Change',
            'place_name': 'State',
            'metric': 'Metric'
        }
    )

    fig.update_layout(
        xaxis_tickangle=-45,
        height=600,
        width=1100
    )

    return fig

Call function then press play on visual

In [22]:
fig = hpi_vs_income_bar_animated()
fig.show()